# Hyperelasticity CVaR Memory Diagnostic

Notebook version of `diagnose_hyperelasticity_cvar_memory.py`.

This notebook repeats evaluation at a fixed control `z` to separate:

- objective-path memory growth: `cost(point, order=0)`
- objective + gradient-path memory growth: `cost(point, order=1)` then `grad(g)`


In [6]:
import os
import sys
import gc
from types import SimpleNamespace

import numpy as np
import soupy

_THIS_DIR = os.path.dirname(os.path.abspath("diagnose_hyperelasticity_cvar_memory.ipynb"))
if _THIS_DIR not in sys.path:
    sys.path.insert(0, _THIS_DIR)

from diagnose_hyperelasticity_cvar_memory import (
    MODEL_CHOICES,
    build_control_np,
    get_current_rss_mb,
    repeat_cost_only,
    repeat_cost1_grad,
    setup_problem,
    make_cvar_cost,
    set_control_part,
)

print("Available models:", MODEL_CHOICES)
print(f"Initial RSS: {get_current_rss_mb():.1f} MB")

Available models: ['linear', 'quadratic', 'mixture_linear_kle', 'mixture_linear_hep', 'mixture_quadratic_kle', 'mixture_quadratic_hep']
Initial RSS: 285.9 MB


In [7]:
# Configuration
args = SimpleNamespace(
    model="mixture_quadratic_kle",
    mode="both",  # one of: cost, grad, both
    repeats=20,
    print_every=1,
    collect_gc=False,
    qoi_type="all",
    penalty=1e-2,
    cvar_beta=0.95,
    n_tr=10,
    n_mix=39,
    quadratic_cvar_n_mc=1000,
    verbose=False,
    control_init="zero",  # one of: zero, constant, random
    control_value=0.5,
    scalar_t=0.0,
    seed=1,
    lx=2.0,
    ly=0.5,
    lz=0.25,
    geometry_dim=2,
    nx=32,
    ny=8,
    nz=12,
)

print(args)

namespace(model='mixture_quadratic_kle', mode='both', repeats=20, print_every=1, collect_gc=False, qoi_type='all', penalty=0.01, cvar_beta=0.95, n_tr=10, n_mix=39, quadratic_cvar_n_mc=1000, verbose=False, control_init='zero', control_value=0.5, scalar_t=0.0, seed=1, lx=2.0, ly=0.5, lz=0.25, geometry_dim=2, nx=32, ny=8, nz=12)


In [8]:
# Build one fixed control vector z in numpy form
base_control_model, _, _ = setup_problem(args)
control_np = build_control_np(base_control_model, args.control_init, args.control_value, args.seed)
del base_control_model
if args.collect_gc:
    gc.collect()

print(f"Control size: {control_np.shape[0]}")
print(f"RSS after control build: {get_current_rss_mb():.1f} MB")

Control size: 297
RSS after control build: 285.9 MB


In [9]:
def run_experiment(label, args, control_np):
    control_model, prior, penalty = setup_problem(args)
    cost = make_cvar_cost(args.model, control_model, prior, penalty, args)
    point = cost.generate_vector(soupy.CONTROL)
    set_control_part(point, control_np, scalar=args.scalar_t)

    print("")
    print(f"=== {label} ===")
    print(f"model={args.model}, repeats={args.repeats}, qoi={args.qoi_type}")
    print(
        f"n_tr={args.n_tr}, n_mix={args.n_mix}, quadratic_cvar_n_mc={args.quadratic_cvar_n_mc}, "
        f"beta={args.cvar_beta}, scalar_t={args.scalar_t}"
    )
    print(f"rss_before_build={get_current_rss_mb():.1f} MB")

    if label == "cost":
        summary = repeat_cost_only(cost, point, args.repeats, args.print_every, args.collect_gc)
    else:
        summary = repeat_cost1_grad(cost, point, args.repeats, args.print_every, args.collect_gc)

    print(
        f"[{summary.label}] summary: start={summary.rss_start_mb:.1f} MB, "
        f"end={summary.rss_end_mb:.1f} MB, max={summary.rss_max_mb:.1f} MB, "
        f"net={summary.rss_end_mb - summary.rss_start_mb:.1f} MB"
    )

    del point
    del cost
    del penalty
    del prior
    del control_model
    if args.collect_gc:
        gc.collect()


In [10]:
if args.mode in ("cost", "both"):
    run_experiment("cost", args, control_np)

if args.mode in ("grad", "both"):
    run_experiment("grad", args, control_np)


=== cost ===
model=mixture_quadratic_kle, repeats=20, qoi=all
n_tr=10, n_mix=39, quadratic_cvar_n_mc=1000, beta=0.95, scalar_t=0.0
rss_before_build=285.9 MB
  [Mixture Quad CVaR][mem] objective start: RSS=285.9 MB
  [Mixture Quad CVaR][mem] after _compute_direction: RSS=291.3 MB
  [Mixture Quad CVaR][mem] after shared_prior construction: RSS=291.3 MB
  [Mixture Quad CVaR][mem] after _build_shared_component_samples: RSS=299.5 MB
  [Mixture Quad CVaR][mem] before explicit old-component cleanup: RSS=299.5 MB
  [Mixture Quad CVaR][mem] after explicit old-component cleanup: RSS=299.5 MB
  [Mixture Quad CVaR][mem] after resetting component containers: RSS=299.5 MB
  [Mixture Quad CVaR][mem] component solver start: RSS=299.5 MB
  [Mixture Quad CVaR][mem] component solver after constructor: RSS=300.4 MB
  [Mixture Quad CVaR][mem] component solver after _linearize_at_mean: RSS=302.1 MB
  [Mixture Quad CVaR][mem] component solver after _compute_eigendecomposition: RSS=316.1 MB
  [Mixture Quad C

KeyboardInterrupt: 